# Forecasting 3 — CRISP-DM Phases 1–6: Business-to-Deployment Workshop

**Business context:** Sportswear retail

**Problem:** Forecast next-week store sales for staffing and stock planning.

**Proposed success criterion:** Beat a four-week rolling-mean baseline MAE by at least 10% on the last 12 weeks; compare stores.

**Dataset:** `forecasting_3.csv`. These are synthetic teaching records, not real company data.

**Submission deadline: Monday 5 October 2026 at 13:00 (1:00 PM).** Submit your completed team notebook on Brightspace. Phases 1–2 must be reviewed and updated; your main practical work is Phases 3–6. Any unfinished in-class work must be completed before the deadline.


## Phase 1 — Business Understanding and Phase 2 — Data Understanding (review)
Explain the decision, stakeholder, prediction moment, observation unit, key and target. **Review or revise the supplied success criterion if it does not adequately represent the business decision.** State two evidenced findings and two limitations.

In [1]:
pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
DATA = Path('forecasting_3.csv')
if not DATA.exists(): DATA = Path('datasets/forecasting_3.csv')
df = pd.read_csv(DATA)
print(df.shape)
display(df.head())
display(df.isna().sum().sort_values(ascending=False).head(12))
print('Exact duplicates:',df.duplicated().sum())

(841, 12)


,store_id,week_start,promo_label,holiday_week,footfall_actual,rain_mm_actual,store_floor_m2,store_size_category,average_price_eur,returns_eur_after_week,sales_eur,list_price_eur
0,S01,01/01/2024,Yes,0,464,13.5,579,Large,14.81,242.75,13098.18,17.18
1,S01,08/01/2024,Yes,0,453,5.0,579,Large,15.25,262.27,14986.99,17.69
2,S01,15/01/2024,Yes,0,445,13.9,579,Large,14.48,485.51,13223.24,16.80
3,S01,22/01/2024,Yes,0,446,16.7,579,Large,14.73,398.76,15302.44,17.09
4,S01,29/01/2024,No,0,395,7.1,579,Large,13.73,180.64,13367.07,15.93


store_id                  0
week_start                0
promo_label               0
holiday_week              0
footfall_actual           0
rain_mm_actual            0
store_floor_m2            0
store_size_category       0
average_price_eur         0
returns_eur_after_week    0
sales_eur                 0
list_price_eur            0
dtype: int64

Exact duplicates: 1


This forecast supports the store manager and regional planner, who use it every week to decide staffing levels and how much stock to request for the following week. The prediction is generated on Monday for the next week, giving managers almost a full week to plan. For example, on Monday of week 10 the model forecasts sales for week 11. Because week 10 has only just started, sales history is only considered complete through week 9. The observation unit is one store in one week, identified by store_id and week_start, and the target is sales_eur.

The proposed success criterion, beating a four week rolling mean baseline by 10 percent MAE, is measurable but incomplete on its own. A single overall MAE can hide the fact that some stores are harder to forecast than others, and MAE treats overestimating and underestimating demand as equally costly, which they are probably not for staffing and stock decisions. We propose keeping the MAE comparison as the main technical measure, but reporting it per store rather than only as one combined number, and separately tracking the share of weeks where demand is underestimated versus overestimated.

Two evidenced findings: the dataset is a complete, balanced weekly panel of 8 stores over 105 weeks each, with no missing values, so coverage is not a concern. We also found one exact duplicate row (store S02, week of 20 May 2024) and one implausible sales value of 82,000 euros for store S03, far above the next highest value of about 18,250 euros, both of which need to be addressed before modelling.

Two limitations: the returns_eur_after_week field is only known after the week is over, so it cannot be used as a predictor without leaking future information into the forecast. Also, promo_label contains four different spellings for what appears to be only two real categories, so the true promotional effect cannot be measured until these labels are cleaned.


## Phase 3 — Data Preparation: three justified decisions
For EACH insight: evidence → technique → before/after verification → interpretation → next decision. Consider identifiers, leakage, duplicate keys, suspicious values, categories, date parsing and feature engineering. Explain why you retain, delete or combine variables. **Do not use the target or post-outcome fields as predictors.**

In [3]:
# Add your preparation code below. Inspect before/after evidence.
# Make a copy first:
work = df.copy()

work['week_start'] = pd.to_datetime(work['week_start'], dayfirst=True)

before = work.shape[0]
work = work.drop_duplicates()
print('Exact duplicates removed:', before - work.shape[0])

print('Before:', work['promo_label'].value_counts())
work['promo_flag'] = work['promo_label'].replace({'Yes': 1, 'promo': 1, 'No': 0, 'REGULAR': 0})
print('After:', work['promo_flag'].value_counts())

print('Correlation:', work[['average_price_eur', 'list_price_eur']].corr().iloc[0, 1])
work = work.drop(columns=['list_price_eur'])

work = work.drop(columns=['returns_eur_after_week'])

store_median = work.groupby('store_id')['sales_eur'].transform('median')
implausible = work['sales_eur'] > 3 * store_median
print('Implausible rows capped:', implausible.sum())
work.loc[implausible, 'sales_eur'] = store_median[implausible]

Exact duplicates removed: 1


Before: promo_label
No         638
Yes        196
REGULAR      4
promo        2
Name: count, dtype: int64
After: promo_flag
0    642
1    198
Name: count, dtype: int64
Correlation: 0.9999904317803365
Implausible rows capped: 1


/tmp/ipykernel_605/138690093.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  work['promo_flag'] = work['promo_label'].replace({'Yes': 1, 'promo': 1, 'No': 0, 'REGULAR': 0})


### Preparation decision log
| Phase 2 finding | Technique and reason | Verification | Risk | Next decision |
|---|---|---|---|---|
|Exact duplicate row (S02, 2024-05-20)|Dropped the duplicate record|Row count 841 to 840|Counting it twice would double that store-week in training|Proceed with the deduplicated panel|
|promo_label had four inconsistent labels|Mapped to a clean binary promo_flag|638/196/4/2 before, clean 642/198 split after|Without this the promotion effect cannot be measured reliably|Use promo_flag as the predictor|
|average_price_eur and list_price_eur correlate at 0.9999|Dropped list_price_eur, kept average_price_eur|Correlation of 0.99999 before, single price field after|Keeping both adds no information and risks instability in linear models|Use average_price_eur only|

We also excluded returns_eur_after_week entirely, since it is only known after the week ends and using it would leak future information into the forecast, and we capped one implausible sales value for store S03 that was roughly four times higher than any other observation for that store.

## Phase 4 — Modelling
Define the prediction moment and allowed predictors. Use a defensible train/test split (chronological for forecasting). Fit learned preprocessing on TRAINING data only. Compare a simple baseline with a candidate model. Explain your feature engineering and excluded fields.

The prediction is generated on Monday for the following week. For example, on Monday of week 10 the model predicts sales for week 11. This gives store managers almost a full week to plan staffing and stock. At that prediction moment, the most recent fully completed sales week is week 9, so sales-history features must use a two-week lag relative to the target week. We can still use information known in advance for the target week, including each store's characteristics (store_id, store_size_category, store_floor_m2), the calendar (holiday_week, month), whether a promotion is planned (promo_flag), and the planned average price (average_price_eur). We do not use footfall_actual or rain_mm_actual for the target week, since both are only measured during that week, at the same time as sales_eur, so including them would leak future information.

We use a chronological split consistent with the forecasting task: for each store, the earliest weeks, once there is enough history to compute the lagged four week rolling average, form the training set, and the last 12 target weeks form the test set, matching the horizon in the proposed success criterion. The sales lag and rolling-average features are shifted by two weeks. This means that a forecast for week 11 made on Monday of week 10 can only use completed sales data through week 9. This keeps the modelling setup aligned with the operational prediction moment.


In [4]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error

model_df = work.sort_values(['store_id', 'week_start']).copy()
model_df['month'] = model_df['week_start'].dt.month

# Monday forecasting setup: when predicting week t, the current week t-1 has only
# just started, so the latest fully completed sales week is t-2.
model_df['sales_lag2'] = model_df.groupby('store_id')['sales_eur'].shift(2)
model_df['sales_roll4_lag2'] = model_df.groupby('store_id')['sales_eur'].transform(
    lambda s: s.shift(2).rolling(4).mean()
)
model_df = model_df.dropna(subset=['sales_lag2', 'sales_roll4_lag2']).reset_index(drop=True)

model_df['rank'] = model_df.groupby('store_id')['week_start'].rank(method='first', ascending=False)
test = model_df[model_df['rank'] <= 12].drop(columns='rank')
train = model_df[model_df['rank'] > 12].drop(columns='rank')
print('Train rows:', train.shape[0], 'Test rows:', test.shape[0])

target = 'sales_eur'
features = ['store_id', 'store_size_category', 'store_floor_m2', 'holiday_week',
            'promo_flag', 'average_price_eur', 'month', 'sales_lag2', 'sales_roll4_lag2']
categorical = ['store_id', 'store_size_category']

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

# Fair Monday baseline: four-week mean using only weeks completed by prediction time.
baseline_pred = test['sales_roll4_lag2']
baseline_mae = mean_absolute_error(y_test, baseline_pred)

pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), categorical)], remainder='passthrough')
candidate = Pipeline([('pre', pre), ('model', RandomForestRegressor(n_estimators=300, random_state=42))])
candidate.fit(X_train, y_train)
candidate_pred = candidate.predict(X_test)
candidate_mae = mean_absolute_error(y_test, candidate_pred)

print('Baseline MAE (four-week rolling mean, Monday setup):', round(baseline_mae, 2))
print('Candidate MAE (Random Forest):', round(candidate_mae, 2))
print('Improvement over baseline:', round((1 - candidate_mae / baseline_mae) * 100, 1), '%')

test = test.copy()
test['baseline_err'] = abs(test['sales_eur'] - baseline_pred)
test['candidate_err'] = abs(test['sales_eur'] - candidate_pred)
print(test.groupby('store_id')[['baseline_err', 'candidate_err']].mean())


Train rows: 704 Test rows: 96


Baseline MAE (four-week rolling mean, Monday setup): 1226.03
Candidate MAE (Random Forest): 837.68
Improvement over baseline: 31.7 %
          baseline_err  candidate_err
store_id                             
S01        1299.307708     924.827947
S02        1008.478542     561.048608
S03        1289.241250     793.684944
S04        1112.041667     744.747897
S05        1498.451667    1217.279150
S06        1230.118125     846.028367
S07        1371.780208     902.538133
S08         998.805208     711.311033


## Phase 5 — Evaluation
Report model vs baseline on held-out observations. Interpret relevant classification errors or forecasting MAE by business segment. Compare with your Phase 1 success criterion and discuss whether that criterion is realistic. If a required cost/impact is unavailable, state the assumption or request more data.

The candidate model reduced the forecast error from 1,226.03 to 837.68 euros, a 31.7 percent improvement over the four week rolling mean baseline under the Monday forecasting setup. This exceeds the 10 percent target proposed in Phase 1, so the technical success criterion was met.

The improvement is also consistent across stores: all eight stores have a lower MAE with the Random Forest than with their store-specific rolling mean baseline. The size of the improvement still differs by store. Store S02 improves the most, from about 1,008 to 561 euros of average error, a 44.4 percent gain, while store S05 has the smallest relative improvement at about 18.8 percent. This confirms why reporting performance per store remains useful even when the overall result is positive.

Looking at the direction of the error, not just its size, the model still systematically underestimates sales in every store. The overall mean signed error is about 397 euros per week, and the model underestimates in about 60 percent of test weeks versus about 40 percent overestimated. This is a business concern because underforecasting can contribute to understaffing and stockouts.

We do not have cost data for staffing or stock, so we cannot yet translate the error reduction into actual money saved. The preparation decisions from Phase 3 helped ensure that the model beats the baseline without relying on information that would not actually be available at forecast time. The Monday prediction setup also makes the forecast more operationally useful by giving managers almost a full week to act. Given that the accuracy target is now met in every store but the model still shows an underestimation bias, we recommend a controlled pilot with monitoring rather than an immediate unmonitored rollout.


In [6]:
test['signed_error'] = test['sales_eur'] - candidate_pred
print(test.groupby('store_id')['signed_error'].mean().round(1))
print()
print('Overall mean signed error:', round(test['signed_error'].mean(), 1))
print('Share of weeks underestimated:', round((test['signed_error'] > 0).mean(), 2))
print('Share of weeks overestimated:', round((test['signed_error'] < 0).mean(), 2))

store_id
S01    421.6
S02    164.3
S03    278.0
S04    440.1
S05    959.6
S06    209.6
S07    335.1
S08    367.9
Name: signed_error, dtype: float64

Overall mean signed error: 397.0
Share of weeks underestimated: 0.6
Share of weeks overestimated: 0.4


## Phase 6 — Deployment proposal (design only)
Specify who uses the prediction, when it runs, what action it supports, a human override, monitoring metric, feedback/refresh trigger and a condition for pausing use. A diagram or 6–8 sentences is sufficient. No production system required.

The store manager and regional planner use this forecast every Monday to predict demand for the following week. For example, on Monday of week 10 the model forecasts week 11 using completed sales history only through week 9, while also using target-week information that is known in advance, such as planned promotions, planned average price and the calendar. This gives managers almost a full week to plan staffing and submit stock replenishment requests before the forecasted week begins. The predicted demand per store supports two decisions: how many staff hours to schedule and how much stock to request from the warehouse. The store manager can override the recommendation when they know about a local event, closure or other information not represented in the model. We would track weekly MAE per store and compare it with the same Monday-compatible four week rolling mean baseline. Although the Random Forest beat the baseline in all eight stores during evaluation, use would be paused for any store if its error exceeds its baseline error for three consecutive weeks, until the team investigates. Because the model still tends to underestimate sales, managers should treat the forecast as a planning aid rather than an exact target and the team should monitor the signed error alongside MAE. No individual customer data is used; the process relies only on store-level weekly data, which limits privacy risk.


## Final group decision + individual accountability
**Group:** Pilot / Improve / Revisit earlier CRISP-DM phase / Do not deploy yet. Cite evidence.

Group: Controlled pilot, not a full rollout.

Under the revised Monday forecasting setup, the candidate model beats the four week rolling mean baseline in all eight stores. Overall MAE falls from 1,226.03 euros to 837.68 euros, a 31.7 percent improvement, which exceeds the 10 percent target from Phase 1. The improvement varies by store, from about 18.8 percent in S05 to 44.4 percent in S02, so store-level monitoring remains important. We recommend a controlled pilot across the stores with weekly comparison against each store's own baseline. This stops short of an unmonitored full rollout because the model still systematically underestimates demand and we lack cost data to confirm how the forecast errors translate into staffing or stock costs.

**Individual:** Every member adds a named Markdown paragraph describing one preparation choice, one model finding and one business implication.

Daniel: One preparation choice I pushed for was dropping list_price_eur instead of average_price_eur, since the two were correlated at 0.9999 and keeping both would have added nothing but redundancy. One finding that stood out to me was that the candidate model consistently underestimates sales rather than making random errors in both directions, which is the more expensive mistake for a store trying to avoid stockouts. The business implication is that even though the revised model now beats the baseline in all eight stores, the underestimation bias still needs to be monitored during a controlled pilot before relying on the forecasts for staffing and stock decisions.

Jayden: One preparation choice I made was cleaning the promo_label variable and turning the different labels into one simple promo_flag, because the dataset used four different labels for what were really only two categories. One model finding I found interesting was that under the revised Monday forecasting setup, the Random Forest improved the MAE from €1,226.03 to €837.68, which is a 31.7% improvement compared with the four-week rolling mean baseline and exceeds the 10% target. The business implication is that the model is promising enough for a controlled pilot, but performance should still be monitored per store because the size of the improvement differs across locations.

Tijn: One preparation choice I made was using a chronological train-test split, because for a forecasting task the model should be trained on earlier weeks and tested on later weeks instead of randomly splitting the data. One model finding I found interesting was that the Random Forest performed differently across stores even though it beat the baseline in all eight, with store S02 showing the largest improvement at about 44.4%. The business implication is that looking only at the overall performance could hide important differences between stores, so forecast accuracy should continue to be monitored separately for each location during the pilot.
